In [ ]:
!pip install -U transformers sentence-transformers peft
import os
os.kill(os.getpid(), 9)


In [ ]:
import re
import random
import os
import json
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

# intents+embeddings
intents = {
    "greeting": ["hi", "hello", "hey", "hii", "heyy", "hellooo"],
    "generic": ["ok", "okay", "sure", "got it", "done", "cool", "noted"],
    "event": [
        "meeting", "class", "exam", "appointment", "birthday", "event",
        "reminder", "schedule", "presentation", "deadline", "submission", "review"
    ],
    "how_are_you": ["how are you", "how have you been", "how are you doing"],
    "availability": ["are you free", "can i call you", "can we talk now", "available now", "are you busy"],
}

intent_embeddings = {k: model.encode(v, convert_to_tensor=True) for k, v in intents.items()}

def classify_message(message):
    msg_emb = model.encode(message, convert_to_tensor=True)
    best_intent, best_score = None, -1

    for intent, emb_list in intent_embeddings.items():
        sim = util.cos_sim(msg_emb, emb_list).max().item()
        if sim > best_score:
            best_intent, best_score = intent, sim

    return best_intent, best_score

def mimic_greeting_intensity(message):
    msg = message.lower().strip()

    if msg.startswith("hey"):
        y_count = len(re.findall(r'y', msg))
        return "he" + "y" * max(1, y_count)

    if msg.startswith("hi"):
        i_count = len(re.findall(r'i', msg))
        return "hi" + "i" * max(1, i_count - 1)

    if msg.startswith("hello"):
        o_count = len(re.findall(r'o', msg))
        return "hello" + "o" * max(1, o_count - 1)

    return random.choice(["hey", "hi", "hello"])

# extracting time and date
def extract_event_details(message):
    msg = message.lower().strip()
    today = datetime.now().date()

    if "day after tomorrow" in msg:
        date_text = (today + timedelta(days=2)).strftime("%Y-%m-%d")
    elif "tomorrow" in msg:
        date_text = (today + timedelta(days=1)).strftime("%Y-%m-%d")
    elif "today" in msg:
        date_text = today.strftime("%Y-%m-%d")
    else:
        weekdays = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
        found_weekday = None

        for i, w in enumerate(weekdays):
            if f"next {w}" in msg:
                found_weekday = i
                today_idx = today.weekday()
                diff = (found_weekday - today_idx) % 7
                if diff == 0:
                    diff = 7
                date_text = (today + timedelta(days=diff)).strftime("%Y-%m-%d")
                break

            if w in msg:
                found_weekday = i
                today_idx = today.weekday()
                diff = (found_weekday - today_idx) % 7
                if diff == 0:
                    diff = 7
                date_text = (today + timedelta(days=diff)).strftime("%Y-%m-%d")
                break

        if found_weekday is None:
            date_match = re.search(r"\b(\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?)", msg)
            if date_match:
                raw = date_match.group(1)
                parts = re.split(r"[/-]", raw)

                try:
                    if len(parts) == 3:
                        date_obj = datetime.strptime(raw, "%d/%m/%Y")
                    else:
                        date_obj = datetime.strptime(raw, "%d/%m")
                        date_obj = date_obj.replace(year=today.year)
                    date_text = date_obj.strftime("%Y-%m-%d")
                except:
                    date_text = "unspecified date"
            else:
                date_text = "unspecified date"
    time_match = re.search(r"(\d{1,2}(:\d{2})?\s*(am|pm)?)", msg)
    time_text = time_match.group(1) if time_match else "09:00"

    return message.strip(), date_text, time_text

def ensure_offline_calendar():
    if not os.path.exists("offline_events.json"):
        with open("offline_events.json", "w", encoding="utf-8") as f:
            json.dump([], f, indent=4)

def event_exists(desc, date_text, time_text):
    ensure_offline_calendar()
    with open("offline_events.json", "r", encoding="utf-8") as f:
        events = json.load(f)

    for ev in events:
        if (
            ev["description"].lower() == desc.lower()
            and ev["date"] == date_text
            and ev["time"] == time_text
        ):
            return True
    return False

def save_to_offline_calendar(desc, date_text, time_text):
    ensure_offline_calendar()

    with open("offline_events.json", "r", encoding="utf-8") as f:
        events = json.load(f)

    events.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "description": desc,
        "date": date_text,
        "time": time_text
    })

    with open("offline_events.json", "w", encoding="utf-8") as f:
        json.dump(events, f, indent=4)

# bot reply
def generate_reply(message):
    intent, score = classify_message(message)

    if score < 0.20:
        return None

    if intent == "greeting":
        return mimic_greeting_intensity(message)

    if intent == "how_are_you":
        return random.choice([
            "I’m good, you?",
            "pretty chill, what about you?",
            "doing fineee :)",
            "all good heree",
            "been okay, how’re you?"
        ])

    if intent == "availability":
        return random.choice([
            "hey sorry, I’m busy rn. I’ll catch up with you later!",
            "sorry, bit tied up now. talk later?",
            "hey, kinda busy atm — ping you soon!",
            "ahh can’t talk rn, will text you later!"
        ])

    if intent == "generic":
        return "yeah"

    if intent == "event":
        desc, date_text, time_text = extract_event_details(message)

        if event_exists(desc, date_text, time_text):
            return None

        save_to_offline_calendar(desc, date_text, time_text)
        return None

    return None

#main program
if __name__ == "__main__":
    print("---- Metro Chatbot ----\n")
    activate = input("Do you want to activate auto-reply for today? (yes/no): ").strip().lower()

    if activate not in ["yes", "y"]:
        print("Auto-reply not activated. See ya later 👋")
    else:
        print("Auto-reply activated! Listening for messages...\n")
        while True:
            msg = input("Incoming message: ")

            if msg.lower() in ["exit", "quit"]:
                break

            if msg.lower() == "show offline calendar":
                ensure_offline_calendar()
                with open("offline_events.json", "r", encoding="utf-8") as f:
                    events = json.load(f)

                if not events:
                    print("\nOffline calendar is empty.\n")
                else:
                    print("\n--- OFFLINE CALENDAR ---")
                    for i, event in enumerate(events, 1):
                        print(f"{i}.")
                        print(f"   Description: {event['description']}")
                        print(f"   Date       : {event['date']}")
                        print(f"   Time       : {event['time']}")
                    print("------------------------\n")
                continue

            reply = generate_reply(msg)

            if reply:
                print(f"Bot reply: {reply}")
            else:
                print("No reply sent.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

---- Metro Chatbot ----

Do you want to activate auto-reply for today? (yes/no): yes
Auto-reply activated! Listening for messages...

Incoming message: hi, how are you?
Bot reply: pretty chill, what about you?
Incoming message: yes im good too
Bot reply: been okay, how’re you?


KeyboardInterrupt: Interrupted by user